In [1]:
import pandas as pd
import numpy as np
import os
import joblib
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer

os.makedirs('../data/processed', exist_ok=True)
os.makedirs('../data/holdout', exist_ok=True)
os.makedirs('../models', exist_ok=True)

### 1. Loading Raw Data and Date Parsing
We load the dataset with latin-1 encoding and parse order date (DateOrders) into datetime objects.

In [3]:
df = pd.read_csv('../DataCoSupplyChainDataset.csv', encoding='latin-1')
df['order_dt'] = pd.to_datetime(df['order date (DateOrders)'])
print(f"Total rows: {len(df)}")
print(f"Date range: {df['order_dt'].min()} to {df['order_dt'].max()}")

Total rows: 180519
Date range: 2015-01-01 00:00:00 to 2018-01-31 23:38:00


### 2. Feature Engineering Justifications
We engineer features based purely on information available at the exact instant an order is confirmed:
1. `order_month` (1-12): Captures seasonal shipping volume changes (e.g. Q4 holiday rush, summer slowdowns) that strain carrier capacities.
2. `order_day` (1-31): Captures month-end fulfillment spikes and inventory rebalancing cycles.
3. `order_dow` (0-6): Day of week (0=Monday, 6=Sunday). Carriers have distinct weekend dispatch schedules and staffing.
4. `order_hour` (0-23): Captures whether an order was placed before or after daily warehouse carrier pickup cutoffs (e.g. afternoon vs night).
5. `is_weekend` (0 or 1): Binary flag indicating orders placed on Saturday or Sunday, which frequently sit in fulfillment queues until Monday.
6. `is_domestic` (0 or 1): Identifies whether shipment destination matches origin country (EE. UU./Estados Unidos or Puerto Rico), as international exports encounter customs checks and intermodal transit steps.
7. `Order Item Discount Rate`: Directly represents promotional markdowns, which often correlate with flash sale volume surges.

In [6]:
df['order_month'] = df['order_dt'].dt.month
df['order_day'] = df['order_dt'].dt.day
df['order_dow'] = df['order_dt'].dt.day_of_week
df['order_hour'] = df['order_dt'].dt.hour
df['is_weekend'] = (df['order_dow'] >= 5).astype(int)

is_us_customer = df['Customer Country'] == 'EE. UU.'
is_us_dest = df['Order Country'] == 'Estados Unidos'
is_pr_customer = df['Customer Country'] == 'Puerto Rico'
is_pr_dest = df['Order Country'] == 'Puerto Rico'
df['is_domestic'] = ((is_us_customer & is_us_dest) | (is_pr_customer & is_pr_dest)).astype(int)

print("Engineered features preview:")
df[['order_month', 'order_day', 'order_dow', 'order_hour', 'is_weekend', 'is_domestic']].head(3)

Engineered features preview:


,order_month,order_day,order_dow,order_hour,is_weekend,is_domestic
0,1,31,2,22,0,0
1,1,13,5,12,1,0
2,1,13,5,12,1,0


### 3. Feature Selection: Order-Time Columns Only
We select features strictly available at order placement and drop all leakage columns (actual shipping days, delivery status, shipping date, post-order status) and non-predictive PII/IDs.

In [7]:
categorical_features = [
    'Shipping Mode',
    'Customer Segment',
    'Market',
    'Order Region',
    'Department Name',
    'Category Name',
    'Type'
]

numerical_features = [
    'Order Item Quantity',
    'Order Item Product Price',
    'Order Item Discount Rate',
    'Order Item Discount',
    'Order Item Total',
    'Sales',
    'order_month',
    'order_day',
    'order_dow',
    'order_hour',
    'is_weekend',
    'is_domestic'
]

target = 'Late_delivery_risk'
feature_cols = categorical_features + numerical_features

print(f"Categorical features ({len(categorical_features)}): {categorical_features}")
print(f"Numerical features ({len(numerical_features)}): {numerical_features}")
print(f"Target: {target}")

Categorical features (7): ['Shipping Mode', 'Customer Segment', 'Market', 'Order Region', 'Department Name', 'Category Name', 'Type']
Numerical features (12): ['Order Item Quantity', 'Order Item Product Price', 'Order Item Discount Rate', 'Order Item Discount', 'Order Item Total', 'Sales', 'order_month', 'order_day', 'order_dow', 'order_hour', 'is_weekend', 'is_domestic']
Target: Late_delivery_risk


### 4. Time-Based Data Split

In [10]:
# Sort chronologically by order timestamp
df_sorted = df.sort_values('order_dt').reset_index(drop=True)

n = len(df_sorted)
train_end = int(0.70 * n)
val_end = int(0.85 * n)

train_df = df_sorted.iloc[:train_end].copy()
val_df = df_sorted.iloc[train_end:val_end].copy()
holdout_df = df_sorted.iloc[val_end:].copy()

print(f"Total dataset: {n:,} rows")
print(f"Train set:     {len(train_df):,} rows ({len(train_df)/n*100:.1f}%) | {train_df['order_dt'].min()} to {train_df['order_dt'].max()} | Late rate: {train_df[target].mean():.4f}")
print(f"Val set:       {len(val_df):,} rows ({len(val_df)/n*100:.1f}%) | {val_df['order_dt'].min()} to {val_df['order_dt'].max()} | Late rate: {val_df[target].mean():.4f}")
print(f"Holdout set:   {len(holdout_df):,} rows ({len(holdout_df)/n*100:.1f}%) | {holdout_df['order_dt'].min()} to {holdout_df['order_dt'].max()} | Late rate: {holdout_df[target].mean():.4f}")

# Save holdout and processed sets
cols_to_save = ['order date (DateOrders)', 'order_dt'] + feature_cols + [target]

train_df[cols_to_save].to_csv('../data/processed/train.csv', index=False)
val_df[cols_to_save].to_csv('../data/processed/val.csv', index=False)
holdout_df[cols_to_save].to_csv('../data/holdout/holdout.csv', index=False)

print("\nDatasets saved to data/processed/ and data/holdout/holdout.csv")

Total dataset: 180,519 rows
Train set:     126,363 rows (70.0%) | 2015-01-01 00:00:00 to 2017-01-08 02:49:00 | Late rate: 0.5494
Val set:       27,078 rows (15.0%) | 2017-01-08 02:49:00 to 2017-06-14 07:19:00 | Late rate: 0.5411
Holdout set:   27,078 rows (15.0%) | 2017-06-14 07:19:00 to 2018-01-31 23:38:00 | Late rate: 0.5502

Datasets saved to data/processed/ and data/holdout/holdout.csv


### 5. Preprocessing Pipeline

In [11]:
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features),
        ('num', StandardScaler(), numerical_features)
    ],
    remainder='drop'
)

# Fit strictly on train
X_train = train_df[feature_cols]
y_train = train_df[target]
preprocessor.fit(X_train)

# Transform train and val
X_train_trans = preprocessor.transform(X_train)
X_val_trans = preprocessor.transform(val_df[feature_cols])

print(f"Raw feature count: {len(feature_cols)}")
print(f"Preprocessed feature count: {X_train_trans.shape[1]}")
print(f"Transformed X_train shape: {X_train_trans.shape}")
print(f"Transformed X_val shape:   {X_val_trans.shape}")

# Save preprocessor
joblib.dump(preprocessor, '../models/preprocessor.joblib')
print("Saved preprocessor to ../models/preprocessor.joblib")

Raw feature count: 19
Preprocessed feature count: 81
Transformed X_train shape: (126363, 81)
Transformed X_val shape:   (27078, 81)
Saved preprocessor to ../models/preprocessor.joblib
